## LPA Analysis with Multiple Variables

Using:
- nt_y_earsapp__mins_mean
- nt_y_earsplt__mins__abcd__social_mean (social media)
- nt_y_earsplt__mins__abcd__stream_mean (streaming/passive consumption)
- nt_y_earsapp__mins__andr__gameaction_mean (gaming) 
- nt_y_earskey__mins_sum (active communication/typing)

In [ ]:
# Ensure that we are using the correct host
import socket
try:
    assert "gpu" in socket.gethostname()
    print(f"Running on {socket.gethostname()}. All is good!")
except:
    raise RuntimeError(f"Be sure to run on GPU! You are currently running on {socket.gethostname()}")

In [ ]:
import subprocess
subprocess.run(["pip", "install", "scikit-learn", "matplotlib", "pandas", "numpy", "seaborn"], check=True)

In [ ]:
from sklearn.mixture import GaussianMixture
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')

Load the Data 

In [ ]:
file_path = r"/shared/healthinfolab/datasets/ABCD/ScreenTime/dataset.tsv/dataset.tsv"
df_raw = pd.read_csv(file_path, sep="\t")

TARGET_VARS = {
    "total_screen_time":  "nt_y_earsapp__mins_mean",
    "social_media":       "nt_y_earsplt__mins__abcd__social_mean",
    "streaming":          "nt_y_earsplt__mins__abcd__stream_mean",
    "gaming":             "nt_y_earsapp__mins__andr__gameaction_mean",
    "keyboard_typing":    "nt_y_earskey__mins_sum",
}

ID_COL      = "participant_id"
SESSION_COL = "session_id"
TARGET_WAVES = ["ses-00A", "ses-01A", "ses-02A", "ses-03A", "ses-04A", "ses-05A"]
WAVE_TO_TIME = {w: i for i, w in enumerate(TARGET_WAVES)}

# Keep only needed columns
keep_cols = [ID_COL, SESSION_COL] + list(TARGET_VARS.values())

# Only keep columns that actually exist in the dataset
keep_cols = [c for c in keep_cols if c in df_raw.columns]
missing = set(TARGET_VARS.values()) - set(df_raw.columns)
if missing:
    print(f"WARNING: These columns were not found and will be skipped: {missing}")

df = df_raw[keep_cols].copy()

# Convert all target vars to numeric and remove negatives
for label, col in TARGET_VARS.items():
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")
        df = df[~(df[col] < 0)]

# Filter to target waves
df = df[df[SESSION_COL].isin(TARGET_WAVES)].copy()
df["time"] = df[SESSION_COL].map(WAVE_TO_TIME)

print(f"Rows after filtering: {len(df)}")
print(f"Unique participants:  {df[ID_COL].nunique()}")
print(f"Columns available:   {[c for c in TARGET_VARS.values() if c in df.columns]}")

Perform per-partiipant linear regression: each participant gets slope + intercept for every screen time dimension (requires at least 2 time stamps)

Feature matrix: `[intercept_total, slope_total, intercept_social, slope_social, ...]`

In [ ]:
MIN_OBSERVATIONS = 2

# Only work with vars present in the data
active_vars = {label: col for label, col in TARGET_VARS.items() if col in df.columns}

records = []

for pid, group in df.groupby(ID_COL):
    row = {"participant_id": pid}
    valid = True

    for label, col in active_vars.items():
        sub = group.dropna(subset=[col])
        if len(sub) < MIN_OBSERVATIONS:
            valid = False
            break
        X = sub["time"].values.reshape(-1, 1)
        y = sub[col].values
        m = LinearRegression().fit(X, y)
        row[f"{label}__intercept"] = m.intercept_
        row[f"{label}__slope"]     = m.coef_[0]

    row["n_obs"] = len(group.dropna(subset=list(active_vars.values()), how="all"))

    if valid:
        records.append(row)

reg_df = pd.DataFrame(records)
print(f"Participants with full regressions: {len(reg_df)}")

# Feature columns = all slope + intercept columns
feature_cols = [c for c in reg_df.columns if c not in ["participant_id", "n_obs"]]
print(f"Feature columns ({len(feature_cols)}): {feature_cols}")

Inspect correlations across all slopes/intercept pairs, ones above 0.7 are flagged 

In [ ]:
corr_matrix = reg_df[feature_cols].corr()

fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix.values, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
plt.colorbar(im, ax=ax)
ticks = range(len(feature_cols))
ax.set_xticks(ticks)
ax.set_yticks(ticks)
ax.set_xticklabels(feature_cols, rotation=45, ha='right', fontsize=8)
ax.set_yticklabels(feature_cols, fontsize=8)
# Annotate cells
for i in range(len(feature_cols)):
    for j in range(len(feature_cols)):
        if i != j:
            ax.text(j, i, f"{corr_matrix.iloc[i, j]:.2f}",
                    ha='center', va='center', fontsize=7,
                    color='white' if abs(corr_matrix.iloc[i, j]) > 0.5 else 'black')
ax.set_title('Feature Correlation Matrix (slopes & intercepts)', fontsize=14, pad=12)
plt.tight_layout()
plt.savefig('correlation_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

# Flag high correlations (|r| > 0.7, excluding diagonal)
high_corr = [
    (feature_cols[i], feature_cols[j], corr_matrix.iloc[i, j])
    for i in range(len(feature_cols))
    for j in range(i + 1, len(feature_cols))
    if abs(corr_matrix.iloc[i, j]) > 0.7
]

if high_corr:
    print('\nHigh correlations (|r| > 0.70) — consider dropping one of each pair:')
    for a, b, r in sorted(high_corr, key=lambda x: abs(x[2]), reverse=True):
        print(f'   {a:45s}  ↔  {b:45s}  r = {r:.2f}')
else:
    print('\nNo feature pairs exceed r = 0.70. All features retained.')


In [ ]:

FEATURES_TO_DROP = []  # default: keep all

# Guard: if feature_cols wasn't set (e.g. previous cell failed), rebuild it
if 'feature_cols' not in dir() or not feature_cols:
    feature_cols = [c for c in reg_df.columns if c not in ['participant_id', 'n_obs', 'LPA_class', 'max_prob']]

final_feature_cols = [c for c in feature_cols if c not in FEATURES_TO_DROP]
print(f'Using {len(final_feature_cols)} features for LPA:')
for f in final_feature_cols:
    print(f'  • {f}')


Standardize features

In [ ]:
scaler = StandardScaler()
features_scaled = scaler.fit_transform(reg_df[final_feature_cols])
features_scaled_df = pd.DataFrame(features_scaled, columns=final_feature_cols)
print("Features standardized. Shape:", features_scaled_df.shape)
print(features_scaled_df.describe().round(2))

Model Selection using BIC and AIC

In [ ]:
results = []

for k in range(1, 7):
    gmm = GaussianMixture(
        n_components=k,
        covariance_type="full",
        random_state=42,
        n_init=10
    )
    gmm.fit(features_scaled_df)
    results.append({
        "Classes": k,
        "BIC":     gmm.bic(features_scaled_df),
        "AIC":     gmm.aic(features_scaled_df),
        "LogLik":  gmm.score(features_scaled_df) * len(features_scaled_df)
    })

results_df = pd.DataFrame(results)
print(results_df.to_string(index=False))

# Plot
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

for ax, metric in zip(axes, ["BIC", "AIC"]):
    ax.plot(results_df["Classes"], results_df[metric], marker="o", linewidth=2)
    best_k = results_df.loc[results_df[metric].idxmin(), "Classes"]
    ax.axvline(best_k, color="red", linestyle="--", alpha=0.6, label=f"Min at k={best_k}")
    ax.set_xlabel("Number of Classes", fontsize=12)
    ax.set_ylabel(metric, fontsize=12)
    ax.set_title(f"Model Selection: {metric}", fontsize=13)
    ax.legend()
    ax.set_xticks(results_df["Classes"])

plt.suptitle("Multivariate LPA — Model Selection", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("model_selection_multivariate.png", dpi=150)
plt.show()

print(f"\nBIC-optimal classes: {results_df.loc[results_df['BIC'].idxmin(), 'Classes']}")
print(f"AIC-optimal classes: {results_df.loc[results_df['AIC'].idxmin(), 'Classes']}")

Final Fit model based on the number of classes from the BIC/AIC plot

In [ ]:
N_CLASSES = 3  # <-- update based on BIC/AIC plot

best_model = GaussianMixture(
    n_components=N_CLASSES,
    covariance_type="full",
    random_state=42,
    n_init=10
)
best_model.fit(features_scaled_df)

reg_df["LPA_class"] = best_model.predict(features_scaled_df)

# Also store posterior probabilities (entropy-based assignment quality)
probs = best_model.predict_proba(features_scaled_df)
reg_df["max_prob"] = probs.max(axis=1)

print("Class distribution:")
print(reg_df["LPA_class"].value_counts().sort_index())
print(f"\nMean posterior probability (assignment certainty): {reg_df['max_prob'].mean():.3f}")
print("(Values closer to 1.0 indicate clean separation between classes)")

Class profile summary

In [ ]:
summary_rows = []
for cls in sorted(reg_df["LPA_class"].unique()):
    sub = reg_df[reg_df["LPA_class"] == cls]
    row = {"Class": cls, "N": len(sub)}
    for label in active_vars:
        row[f"{label} intercept"] = sub[f"{label}__intercept"].mean()
        row[f"{label} slope"]     = sub[f"{label}__slope"].mean()
    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows).set_index("Class")

# Display intercepts
print("=== Mean Intercepts (Baseline Level at Wave 0) ===")
intercept_cols = [c for c in summary_df.columns if "intercept" in c]
print(summary_df[["N"] + intercept_cols].round(2).to_string())

print("\n=== Mean Slopes (Change per Wave) ===")
slope_cols = [c for c in summary_df.columns if "slope" in c]
print(summary_df[["N"] + slope_cols].round(2).to_string())

Create Mean Trajectory Plots

In [ ]:
time_points  = np.arange(len(TARGET_WAVES))
var_labels   = list(active_vars.keys())
n_vars       = len(var_labels)
colors       = plt.cm.tab10.colors
classes      = sorted(reg_df["LPA_class"].unique())

fig, axes = plt.subplots(1, n_vars, figsize=(5 * n_vars, 5), sharey=False)
if n_vars == 1:
    axes = [axes]

for ax, label in zip(axes, var_labels):
    for cls in classes:
        sub  = reg_df[reg_df["LPA_class"] == cls]
        n    = len(sub)
        traj = sub[f"{label}__intercept"].mean() + sub[f"{label}__slope"].mean() * time_points
        ax.plot(
            time_points, traj,
            marker="o",
            color=colors[cls % len(colors)],
            linewidth=2,
            label=f"Class {cls} (n={n}, slope={sub[f'{label}__slope'].mean():.1f})"
        )
    ax.set_title(label.replace("_", " ").title(), fontsize=11)
    ax.set_xticks(time_points)
    ax.set_xticklabels(TARGET_WAVES, rotation=30, fontsize=8)
    ax.set_xlabel("Wave")
    ax.legend(fontsize=8)

axes[0].set_ylabel("Mean Screen Time (mins)")
fig.suptitle("Mean Trajectory by LPA Class — All Variables", fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig("trajectory_classes_multivariate.png", dpi=150, bbox_inches="tight")
plt.show()

Individual trajectories by class (for the total screen time variable)

In [ ]:
PLOT_VAR = "total_screen_time"  # <-- change to any key in TARGET_VARS

fig, axes = plt.subplots(1, len(classes), figsize=(5 * len(classes), 5), sharey=True)
if len(classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, classes):
    sub   = reg_df[reg_df["LPA_class"] == cls]
    color = colors[cls % len(colors)]

    for _, row in sub.iterrows():
        y_ind = row[f"{PLOT_VAR}__intercept"] + row[f"{PLOT_VAR}__slope"] * time_points
        ax.plot(time_points, y_ind, color=color, alpha=0.07, linewidth=0.8, zorder=1)

    mean_traj = sub[f"{PLOT_VAR}__intercept"].mean() + sub[f"{PLOT_VAR}__slope"].mean() * time_points
    ax.plot(
        time_points, mean_traj,
        color=color, linewidth=3, marker="o", markersize=7, zorder=2,
        label=f"Mean (slope={sub[f'{PLOT_VAR}__slope'].mean():.1f})"
    )
    ax.set_title(f"Class {cls}  (n={len(sub)})", fontsize=12)
    ax.set_xticks(time_points)
    ax.set_xticklabels(TARGET_WAVES, rotation=30, fontsize=8)
    ax.set_xlabel("Wave")
    ax.legend(fontsize=9)

axes[0].set_ylabel(f"{PLOT_VAR.replace('_', ' ').title()} (mins)")
fig.suptitle(f"Individual Trajectories by LPA Class — {PLOT_VAR.replace('_', ' ').title()}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"individual_trajectories_{PLOT_VAR}.png", dpi=150, bbox_inches="tight")
plt.show()

Visualize how each class differs across all 5 screen time dimensions (using standardized intercepts so scales are comparable)

In [ ]:
PLOT_VAR = "total_screen_time"  # <-- change to any key in TARGET_VARS

fig, axes = plt.subplots(1, len(classes), figsize=(5 * len(classes), 5), sharey=True)
if len(classes) == 1:
    axes = [axes]

for ax, cls in zip(axes, classes):
    sub   = reg_df[reg_df["LPA_class"] == cls]
    color = colors[cls % len(colors)]

    for _, row in sub.iterrows():
        y_ind = row[f"{PLOT_VAR}__intercept"] + row[f"{PLOT_VAR}__slope"] * time_points
        ax.plot(time_points, y_ind, color=color, alpha=0.07, linewidth=0.8, zorder=1)

    mean_traj = sub[f"{PLOT_VAR}__intercept"].mean() + sub[f"{PLOT_VAR}__slope"].mean() * time_points
    ax.plot(
        time_points, mean_traj,
        color=color, linewidth=3, marker="o", markersize=7, zorder=2,
        label=f"Mean (slope={sub[f'{PLOT_VAR}__slope'].mean():.1f})"
    )
    ax.set_title(f"Class {cls}  (n={len(sub)})", fontsize=12)
    ax.set_xticks(time_points)
    ax.set_xticklabels(TARGET_WAVES, rotation=30, fontsize=8)
    ax.set_xlabel("Wave")
    ax.legend(fontsize=9)

axes[0].set_ylabel(f"{PLOT_VAR.replace('_', ' ').title()} (mins)")
fig.suptitle(f"Individual Trajectories by LPA Class — {PLOT_VAR.replace('_', ' ').title()}",
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f"individual_trajectories_{PLOT_VAR}.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
output_cols = ["participant_id", "n_obs", "LPA_class", "max_prob"] + feature_cols
output_df = reg_df[output_cols]
output_df.to_csv("longitudinal_multivariate_lpa.csv", index=False)

print("Saved to longitudinal_multivariate_lpa.csv")
print(output_df.head())